
# EfficientNet-B0 Eye ROI Baseline — Revised Stable Version

Bu notebook doğrudan şu proje yapısına göre hazırlanmıştır:

- **Input:** `AISC DeepFake Çalışmaları / Deneyler / Kader / Deney 1 / Göz / eye_roi_output`
- **Metadata:** `eye_roi_output/metadata.csv`
- **Output:** `AISC DeepFake Çalışmaları / Deneyler / Kader / Deney 1 / Sonuçlar/<run_id>`

Temel ilkeler:
- mevcut `train/val/test` splitleri korunur,
- `no_face` gibi audit satırları eğitime alınmaz,
- frozen stage + full fine-tuning uygulanır,
- fine-tuning aşamasında sayısal kararlılık için FP32 kullanılır,
- checkpoint resume güvenli yapılır,
- validation threshold seçimi yalnızca validation setinde yapılır,
- test seti final değerlendirmede kullanılır.


In [1]:

# ============================================================
# CELL 1 — IMPORTS
# ============================================================

import os
import gc
import json
import math
import time
import random
import hashlib
import platform
import subprocess
import sys
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

import matplotlib.pyplot as plt

ImageFile.LOAD_TRUNCATED_IMAGES = False

print("Python:", platform.python_version())
print("Torch :", torch.__version__)
print("CUDA  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))


Python: 3.12.13
Torch : 2.11.0+cu128
CUDA  : True
GPU   : Tesla T4


In [2]:

# ============================================================
# CELL 2 — CONFIG
# ============================================================

@dataclass(frozen=True)
class Config:
    seed: int = 42
    image_size: int = 224

    batch_size: int = 32
    num_workers: int = 2

    frozen_epochs: int = 5
    finetune_epochs: int = 15

    frozen_lr: float = 1e-3
    finetune_lr: float = 2e-5

    weight_decay: float = 1e-4
    patience: int = 4
    grad_clip_norm: float = 1.0

    dropout: float = 0.2

    threshold_min: float = 0.05
    threshold_max: float = 0.95
    threshold_steps: int = 181

    accepted_statuses: Tuple[str, ...] = ("ok", "success")
    positive_label: str = "fake"
    negative_label: str = "real"

    resume: bool = True

CONFIG = Config()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(CONFIG)
print("DEVICE:", DEVICE)


Config(seed=42, image_size=224, batch_size=32, num_workers=2, frozen_epochs=5, finetune_epochs=15, frozen_lr=0.001, finetune_lr=2e-05, weight_decay=0.0001, patience=4, grad_clip_norm=1.0, dropout=0.2, threshold_min=0.05, threshold_max=0.95, threshold_steps=181, accepted_statuses=('ok', 'success'), positive_label='fake', negative_label='real', resume=True)
DEVICE: cuda


In [3]:

# ============================================================
# CELL 3 — REPRODUCIBILITY
# ============================================================

def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(True, warn_only=True)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


seed_everything(CONFIG.seed)
print("Seed fixed:", CONFIG.seed)


Seed fixed: 42


In [4]:

# ============================================================
# CELL 4 — MOUNT GOOGLE DRIVE
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError as e:
    raise RuntimeError(
        "This notebook is intended to run in Google Colab."
    ) from e

assert Path("/content/drive/MyDrive").exists(), "Google Drive mount failed."

print("Google Drive mounted.")


Mounted at /content/drive
Google Drive mounted.


In [5]:

# ============================================================
# CELL 5 — FIXED PROJECT PATHS
# ============================================================

DENEY1_ROOT = Path(
    "/content/drive/MyDrive/"
    "AISC DeepFake Çalışmaları/"
    "Deneyler/"
    "Kader/"
    "Deney 1"
)

GOZ_ROOT = DENEY1_ROOT / "Göz"
ROI_ROOT = GOZ_ROOT / "eye_roi_output"
METADATA_PATH = ROI_ROOT / "metadata.csv"

# IMPORTANT:
# Results are directly under Kader / Deney 1 / Sonuçlar
RESULTS_ROOT = DENEY1_ROOT / "Sonuçlar"

for p, name in [
    (DENEY1_ROOT, "DENEY1_ROOT"),
    (GOZ_ROOT, "GOZ_ROOT"),
    (ROI_ROOT, "ROI_ROOT"),
]:
    if not p.is_dir():
        raise FileNotFoundError(f"{name} not found:\n{p}")

if not METADATA_PATH.is_file():
    raise FileNotFoundError(f"metadata.csv not found:\n{METADATA_PATH}")

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PATH CONFIGURATION OK")
print("=" * 80)
print("DENEY1_ROOT :", DENEY1_ROOT)
print("GOZ_ROOT    :", GOZ_ROOT)
print("ROI_ROOT    :", ROI_ROOT)
print("METADATA    :", METADATA_PATH)
print("RESULTS_ROOT:", RESULTS_ROOT)


PATH CONFIGURATION OK
DENEY1_ROOT : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1
GOZ_ROOT    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz
ROI_ROOT    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output
METADATA    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/metadata.csv
RESULTS_ROOT: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar


In [6]:

# ============================================================
# CELL 6 — RUN DIRECTORY
# ============================================================

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M") + "_eye_efficientnet_b0_seed42"
RUN_DIR = RESULTS_ROOT / RUN_ID

suffix = 1
while RUN_DIR.exists():
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M") + f"_eye_efficientnet_b0_seed42_r{suffix}"
    RUN_DIR = RESULTS_ROOT / RUN_ID
    suffix += 1

DIRS = {
    "checkpoints": RUN_DIR / "checkpoints",
    "metrics": RUN_DIR / "metrics",
    "predictions": RUN_DIR / "predictions",
    "figures": RUN_DIR / "figures",
    "artifacts": RUN_DIR / "artifacts",
    "logs": RUN_DIR / "logs",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

FROZEN_DIR = DIRS["checkpoints"] / "frozen"
FINETUNE_DIR = DIRS["checkpoints"] / "finetune"
FROZEN_DIR.mkdir(parents=True, exist_ok=True)
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_ID :", RUN_ID)
print("RUN_DIR:", RUN_DIR)


RUN_ID : 20260808_0747_eye_efficientnet_b0_seed42
RUN_DIR: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260808_0747_eye_efficientnet_b0_seed42


In [7]:

# ============================================================
# CELL 7 — ATOMIC I/O HELPERS
# ============================================================

def atomic_write_bytes(data: bytes, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + ".tmp")

    with open(tmp, "wb") as f:
        f.write(data)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, target)


def atomic_write_text(text: str, target: Path, encoding: str = "utf-8") -> None:
    atomic_write_bytes(text.encode(encoding), target)


def atomic_json_dump(obj: Any, target: Path) -> None:
    atomic_write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        target,
    )


def atomic_yaml_dump(obj: Any, target: Path) -> None:
    atomic_write_text(
        yaml.safe_dump(obj, sort_keys=False, allow_unicode=True),
        target,
    )


def atomic_csv_dump(df: pd.DataFrame, target: Path) -> None:
    atomic_write_bytes(df.to_csv(index=False).encode("utf-8"), target)


def atomic_torch_save(state: dict, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)

    tmp = target.with_suffix(target.suffix + ".tmp")
    torch.save(state, tmp)

    # Verify checkpoint is readable before replacing target.
    verify = torch.load(tmp, map_location="cpu", weights_only=False)

    required = {
        "epoch",
        "model_state_dict",
        "optimizer_state_dict",
    }

    if not required.issubset(verify.keys()):
        try:
            tmp.unlink()
        finally:
            raise RuntimeError(f"Checkpoint validation failed: {tmp}")

    os.replace(tmp, target)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


atomic_yaml_dump(asdict(CONFIG), RUN_DIR / "config_resolved.yaml")

atomic_json_dump(
    {
        "run_id": RUN_ID,
        "python": platform.python_version(),
        "torch": torch.__version__,
        "device": str(DEVICE),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
    RUN_DIR / "environment.json",
)

try:
    freeze = subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
    )
    atomic_write_text(freeze, RUN_DIR / "requirements_lock.txt")
except Exception as e:
    atomic_write_text(
        f"pip freeze failed: {repr(e)}\n",
        RUN_DIR / "requirements_lock.txt",
    )

print("Atomic I/O helpers ready.")


Atomic I/O helpers ready.


In [8]:

# ============================================================
# CELL 8 — LOAD + VALIDATE METADATA
# ============================================================

REQUIRED_METADATA_COLUMNS = {
    "sample_id",
    "label",
    "split",
    "status",
    "combined_eye_path",
}

metadata_raw = pd.read_csv(
    METADATA_PATH,
    encoding="utf-8-sig",
)

metadata_raw.columns = [
    str(c).strip()
    for c in metadata_raw.columns
]

missing_columns = REQUIRED_METADATA_COLUMNS.difference(metadata_raw.columns)

if missing_columns:
    raise ValueError(
        f"metadata.csv is missing required columns: {sorted(missing_columns)}"
    )

metadata = metadata_raw.copy()

# Preserve real missing values instead of converting them to literal "nan".
for col in [
    "sample_id",
    "label",
    "split",
    "status",
    "combined_eye_path",
]:
    metadata[col] = metadata[col].astype("string").str.strip()

metadata["label"] = metadata["label"].str.lower()
metadata["split"] = metadata["split"].str.lower()
metadata["status"] = metadata["status"].str.lower()

allowed_labels = {
    CONFIG.negative_label,
    CONFIG.positive_label,
}

allowed_splits = {"train", "val", "test"}
accepted_statuses = {str(x).lower() for x in CONFIG.accepted_statuses}

observed_labels = set(metadata["label"].dropna().unique())
observed_splits = set(metadata["split"].dropna().unique())

bad_labels = sorted(observed_labels - allowed_labels)
bad_splits = sorted(observed_splits - allowed_splits)

if bad_labels:
    raise ValueError(f"Unexpected labels: {bad_labels}")

if bad_splits:
    raise ValueError(f"Unexpected splits: {bad_splits}")

status_counts = (
    metadata["status"]
    .fillna("<missing>")
    .value_counts(dropna=False)
)

print("Status counts:")
print(status_counts)

accepted_mask = metadata["status"].isin(accepted_statuses)

audit_only = metadata.loc[~accepted_mask].copy()
accepted = metadata.loc[accepted_mask].copy()

if accepted.empty:
    raise RuntimeError("No accepted eye ROI rows exist.")

# Accepted rows must have sample_id.
missing_sample = accepted["sample_id"].isna() | accepted["sample_id"].eq("")

if missing_sample.any():
    raise ValueError(
        "Accepted rows with missing sample_id were found."
    )

duplicate_mask = accepted["sample_id"].duplicated(keep=False)

if duplicate_mask.any():
    dupes = (
        accepted.loc[duplicate_mask, "sample_id"]
        .head(20)
        .tolist()
    )
    raise ValueError(
        f"Duplicate sample_id values among accepted rows: {dupes}"
    )

missing_combined = (
    accepted["combined_eye_path"].isna()
    | accepted["combined_eye_path"].eq("")
)

if missing_combined.any():
    raise ValueError(
        "Accepted rows with missing combined_eye_path were found."
    )


def resolve_roi_path(value: str) -> Path:
    p = Path(str(value))

    if p.is_absolute():
        return p

    return ROI_ROOT / p


accepted["resolved_eye_path"] = (
    accepted["combined_eye_path"]
    .map(resolve_roi_path)
)

accepted["path_exists"] = (
    accepted["resolved_eye_path"]
    .map(lambda p: p.is_file())
)

missing_file_rows = accepted.loc[~accepted["path_exists"]].copy()

if len(missing_file_rows) > 0:
    print("WARNING: accepted rows with missing image files:", len(missing_file_rows))
    print(
        missing_file_rows[
            ["sample_id", "combined_eye_path", "resolved_eye_path"]
        ].head(20)
    )

eligible = (
    accepted.loc[accepted["path_exists"]]
    .copy()
    .reset_index(drop=True)
)

if eligible.empty:
    raise RuntimeError("No eligible eye ROI images were found.")

# Verify every split contains both classes.
ct = pd.crosstab(
    eligible["split"],
    eligible["label"],
)

print("\nEligible split / class counts:")
print(ct)

for split in ["train", "val", "test"]:
    split_rows = eligible.loc[eligible["split"] == split]

    if split_rows.empty:
        raise ValueError(f"Required split is absent: {split}")

    split_labels = set(split_rows["label"].dropna().unique())

    if split_labels != allowed_labels:
        raise ValueError(
            f"Split {split!r} must contain both classes. Found: {split_labels}"
        )

# Path-level split leakage check.
split_paths = {
    split: set(
        eligible.loc[
            eligible["split"] == split,
            "resolved_eye_path",
        ].astype(str)
    )
    for split in ["train", "val", "test"]
}

path_intersections = {
    "train_val": len(split_paths["train"] & split_paths["val"]),
    "train_test": len(split_paths["train"] & split_paths["test"]),
    "val_test": len(split_paths["val"] & split_paths["test"]),
}

if any(path_intersections.values()):
    raise ValueError(
        f"Cross-split duplicate ROI paths detected: {path_intersections}"
    )

data_accounting = {
    "run_id": RUN_ID,
    "metadata_path": str(METADATA_PATH),
    "total_metadata_rows": int(len(metadata)),
    "accepted_status_rows": int(len(accepted)),
    "audit_only_rows": int(len(audit_only)),
    "missing_files_among_accepted": int(len(missing_file_rows)),
    "training_eligible_success_count": int(len(eligible)),
    "status_counts": {
        str(k): int(v)
        for k, v in status_counts.items()
    },
    "split_class_counts": {
        split: {
            label: int(
                (
                    (eligible["split"] == split)
                    & (eligible["label"] == label)
                ).sum()
            )
            for label in sorted(allowed_labels)
        }
        for split in ["train", "val", "test"]
    },
    "cross_split_path_intersections": path_intersections,
    "true_video_level_leakage_status": "NOT_VERIFIABLE_FROM_CURRENT_METADATA",
    "true_video_level_note": (
        "Current video_id is not treated as authoritative original source-video identity."
    ),
}

atomic_json_dump(
    data_accounting,
    RUN_DIR / "data_accounting.json",
)

atomic_csv_dump(
    eligible,
    DIRS["artifacts"] / "eligible_metadata.csv",
)

atomic_csv_dump(
    audit_only,
    DIRS["artifacts"] / "skipped_metadata.csv",
)

print("\nMETADATA VALIDATION PASSED")
print(json.dumps(data_accounting, indent=2, ensure_ascii=False))


Status counts:
status
ok         2986
no_face     111
Name: count, dtype: Int64

Eligible split / class counts:
label  fake  real
split            
test    156   146
train  1191  1197
val     141   155

METADATA VALIDATION PASSED
{
  "run_id": "20260808_0747_eye_efficientnet_b0_seed42",
  "metadata_path": "/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/metadata.csv",
  "total_metadata_rows": 3097,
  "accepted_status_rows": 2986,
  "audit_only_rows": 111,
  "missing_files_among_accepted": 0,
  "training_eligible_success_count": 2986,
  "status_counts": {
    "ok": 2986,
    "no_face": 111
  },
  "split_class_counts": {
    "train": {
      "fake": 1191,
      "real": 1197
    },
    "val": {
      "fake": 141,
      "real": 155
    },
    "test": {
      "fake": 156,
      "real": 146
    }
  },
  "cross_split_path_intersections": {
    "train_val": 0,
    "train_test": 0,
    "val_test": 0
  },
  "true_video_level_leakage_status": "NOT_VERIFI

In [9]:

# ============================================================
# CELL 9 — TRANSFORMS + DATASETS + DATALOADERS
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(
        (CONFIG.image_size, CONFIG.image_size),
        interpolation=transforms.InterpolationMode.BICUBIC,
        antialias=True,
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(
        (CONFIG.image_size, CONFIG.image_size),
        interpolation=transforms.InterpolationMode.BICUBIC,
        antialias=True,
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

LABEL_TO_INT = {
    CONFIG.negative_label: 0,
    CONFIG.positive_label: 1,
}

INT_TO_LABEL = {
    0: CONFIG.negative_label,
    1: CONFIG.positive_label,
}


class EyeROIDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform):
        self.df = frame.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        path = Path(row["resolved_eye_path"])

        try:
            with Image.open(path) as img:
                img = img.convert("RGB")
                image = self.transform(img)
        except Exception as e:
            raise RuntimeError(
                f"Failed to read image at index={idx}, path={path}"
            ) from e

        label = torch.tensor(
            LABEL_TO_INT[row["label"]],
            dtype=torch.float32,
        )

        return {
            "image": image,
            "label": label,
            "sample_id": str(row["sample_id"]),
            "path": str(path),
        }


split_frames = {
    split: (
        eligible.loc[eligible["split"] == split]
        .copy()
        .reset_index(drop=True)
    )
    for split in ["train", "val", "test"]
}

datasets = {
    "train": EyeROIDataset(split_frames["train"], train_transform),
    "val": EyeROIDataset(split_frames["val"], eval_transform),
    "test": EyeROIDataset(split_frames["test"], eval_transform),
}

loader_generator = torch.Generator()
loader_generator.manual_seed(CONFIG.seed)

loaders = {
    "train": DataLoader(
        datasets["train"],
        batch_size=CONFIG.batch_size,
        shuffle=True,
        num_workers=CONFIG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(CONFIG.num_workers > 0),
        generator=loader_generator,
        drop_last=False,
    ),
    "val": DataLoader(
        datasets["val"],
        batch_size=CONFIG.batch_size,
        shuffle=False,
        num_workers=CONFIG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(CONFIG.num_workers > 0),
        drop_last=False,
    ),
    "test": DataLoader(
        datasets["test"],
        batch_size=CONFIG.batch_size,
        shuffle=False,
        num_workers=CONFIG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(CONFIG.num_workers > 0),
        drop_last=False,
    ),
}

print({
    split: len(loader.dataset)
    for split, loader in loaders.items()
})

sample_batch = next(iter(loaders["train"]))

assert sample_batch["image"].shape[1:] == (
    3,
    CONFIG.image_size,
    CONFIG.image_size,
)

print("Batch shape:", tuple(sample_batch["image"].shape))


{'train': 2388, 'val': 296, 'test': 302}
Batch shape: (32, 3, 224, 224)


In [10]:

# ============================================================
# CELL 10 — MODEL
# ============================================================

def build_model(pretrained: bool = True) -> nn.Module:
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None

    model = efficientnet_b0(weights=weights)

    in_features = model.classifier[1].in_features

    model.classifier = nn.Sequential(
        nn.Dropout(p=CONFIG.dropout),
        nn.Linear(in_features, 1),
    )

    return model


model = build_model(pretrained=True).to(DEVICE)

print(
    "Total params:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

print(
    "Trainable params:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 215MB/s]


Total params: 4,008,829
Trainable params: 4,008,829


In [11]:

# ============================================================
# CELL 11 — METRICS
# ============================================================

def safe_roc_auc(y_true, probs):
    if len(np.unique(y_true)) < 2:
        return float("nan")

    return float(
        roc_auc_score(
            y_true,
            probs,
        )
    )


def safe_ap(y_true, probs):
    if len(np.unique(y_true)) < 2:
        return float("nan")

    return float(
        average_precision_score(
            y_true,
            probs,
        )
    )


def binary_metrics(
    y_true,
    probs,
    threshold: float,
) -> Dict[str, float]:

    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probs = np.asarray(
        probs,
        dtype=float,
    )

    preds = (
        probs >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        preds,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else float("nan")
    )

    return {
        "threshold": float(threshold),
        "accuracy": float(
            accuracy_score(y_true, preds)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, preds)
        ),
        "precision": float(
            precision_score(
                y_true,
                preds,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                preds,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                preds,
                zero_division=0,
            )
        ),
        "specificity": float(specificity),
        "roc_auc": safe_roc_auc(
            y_true,
            probs,
        ),
        "average_precision": safe_ap(
            y_true,
            probs,
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def select_threshold_on_validation(y_true, probs):
    thresholds = np.linspace(
        CONFIG.threshold_min,
        CONFIG.threshold_max,
        CONFIG.threshold_steps,
    )

    rows = [
        binary_metrics(
            y_true,
            probs,
            float(t),
        )
        for t in thresholds
    ]

    table = pd.DataFrame(rows)

    table["distance_to_0_5"] = (
        table["threshold"] - 0.5
    ).abs()

    best = table.sort_values(
        [
            "f1",
            "balanced_accuracy",
            "distance_to_0_5",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    ).iloc[0]

    return (
        float(best["threshold"]),
        table.drop(
            columns=["distance_to_0_5"]
        ),
    )


In [12]:

# ============================================================
# CELL 12 — SAFE CHECKPOINT HELPERS
# ============================================================

def get_rng_state() -> dict:
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state().cpu(),
    }

    if torch.cuda.is_available():
        state["cuda"] = [
            s.cpu()
            for s in torch.cuda.get_rng_state_all()
        ]

    return state


def set_rng_state(state: Optional[dict]) -> None:
    if not state:
        return

    if state.get("python") is not None:
        random.setstate(state["python"])

    if state.get("numpy") is not None:
        np.random.set_state(state["numpy"])

    torch_state = state.get("torch")

    if torch_state is not None:
        if not torch.is_tensor(torch_state):
            torch_state = torch.tensor(
                torch_state,
                dtype=torch.uint8,
            )

        torch.set_rng_state(
            torch_state
            .detach()
            .cpu()
            .to(torch.uint8)
        )

    cuda_states = state.get("cuda")

    if (
        torch.cuda.is_available()
        and cuda_states is not None
    ):
        safe_cuda_states = []

        for s in cuda_states:
            if not torch.is_tensor(s):
                s = torch.tensor(
                    s,
                    dtype=torch.uint8,
                )

            safe_cuda_states.append(
                s
                .detach()
                .cpu()
                .to(torch.uint8)
            )

        if len(safe_cuda_states) == torch.cuda.device_count():
            torch.cuda.set_rng_state_all(
                safe_cuda_states
            )


def save_checkpoint(
    path: Path,
    *,
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    best_score: float,
    history: List[dict],
    stage_name: str,
) -> None:

    state = {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": (
            scheduler.state_dict()
            if scheduler is not None
            else None
        ),
        "best_metric_score": float(best_score),
        "history": history,
        "stage_name": stage_name,
        "config": asdict(CONFIG),
        "rng_state": get_rng_state(),
    }

    atomic_torch_save(
        state,
        path,
    )


def load_checkpoint(
    path: Path,
    *,
    model: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler=None,
) -> dict:

    # Load on CPU first to avoid RNG state device problems.
    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    model.load_state_dict(
        ckpt["model_state_dict"]
    )

    model.to(DEVICE)

    if optimizer is not None:
        optimizer.load_state_dict(
            ckpt["optimizer_state_dict"]
        )

        # Move optimizer tensors to current device.
        for state in optimizer.state.values():
            for key, value in state.items():
                if torch.is_tensor(value):
                    state[key] = value.to(DEVICE)

    if (
        scheduler is not None
        and ckpt.get("scheduler_state_dict") is not None
    ):
        scheduler.load_state_dict(
            ckpt["scheduler_state_dict"]
        )

    set_rng_state(
        ckpt.get("rng_state")
    )

    return ckpt


In [13]:

# ============================================================
# CELL 13 — SAFE EPOCH RUNNER
# ============================================================

criterion = nn.BCEWithLogitsLoss()


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    *,
    training: bool,
    optimizer: Optional[torch.optim.Optimizer] = None,
) -> dict:

    if training:
        if optimizer is None:
            raise ValueError(
                "optimizer is required when training=True"
            )
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    n_items = 0

    all_labels = []
    all_probs = []
    all_sample_ids = []
    all_paths = []

    for batch_idx, batch in enumerate(loader):

        images = batch["image"].to(
            DEVICE,
            non_blocking=True,
        )

        labels = batch["label"].to(
            DEVICE,
            non_blocking=True,
        )

        if not torch.isfinite(images).all():
            raise FloatingPointError(
                f"NaN/Inf detected in images at batch={batch_idx}"
            )

        if not torch.isfinite(labels).all():
            raise FloatingPointError(
                f"NaN/Inf detected in labels at batch={batch_idx}"
            )

        if training:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(training):

            # Full FP32 on purpose for numerical stability.
            logits = (
                model(images)
                .squeeze(1)
                .float()
            )

            if not torch.isfinite(logits).all():
                raise FloatingPointError(
                    f"Non-finite logits at batch={batch_idx}"
                )

            loss = criterion(
                logits,
                labels.float(),
            )

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite loss at batch={batch_idx}"
                )

            if training:
                loss.backward()

                bad_grad_names = []

                for name, param in model.named_parameters():
                    if param.grad is None:
                        continue

                    if not torch.isfinite(param.grad).all():
                        bad_grad_names.append(name)

                if bad_grad_names:
                    raise FloatingPointError(
                        "Non-finite gradients detected at "
                        f"batch={batch_idx}. "
                        f"First affected params: {bad_grad_names[:10]}"
                    )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    CONFIG.grad_clip_norm,
                    error_if_nonfinite=True,
                )

                optimizer.step()

        probs = torch.sigmoid(
            logits.detach()
        )

        if not torch.isfinite(probs).all():
            raise FloatingPointError(
                f"Non-finite probabilities at batch={batch_idx}"
            )

        batch_size = images.size(0)

        total_loss += (
            float(loss.detach().item())
            * batch_size
        )

        n_items += batch_size

        all_labels.extend(
            labels
            .detach()
            .cpu()
            .numpy()
            .astype(int)
            .tolist()
        )

        all_probs.extend(
            probs
            .cpu()
            .numpy()
            .astype(float)
            .tolist()
        )

        all_sample_ids.extend(
            list(batch["sample_id"])
        )

        all_paths.extend(
            list(batch["path"])
        )

    if n_items != len(loader.dataset):
        raise RuntimeError(
            f"Epoch accounting mismatch: {n_items} != {len(loader.dataset)}"
        )

    return {
        "loss": total_loss / max(n_items, 1),
        "labels": np.asarray(all_labels, dtype=int),
        "probabilities": np.asarray(all_probs, dtype=float),
        "sample_ids": all_sample_ids,
        "paths": all_paths,
        "count": int(n_items),
    }


In [14]:

# ============================================================
# CELL 14 — SMOKE TEST
# ============================================================

def smoke_test(model: nn.Module) -> None:
    print("Running 2-batch smoke test...")

    backup = {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }

    temp_optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-6,
    )

    model.train()

    seen = 0

    for batch in loaders["train"]:

        images = batch["image"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        temp_optimizer.zero_grad(
            set_to_none=True
        )

        logits = (
            model(images)
            .squeeze(1)
            .float()
        )

        loss = criterion(
            logits,
            labels.float(),
        )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                "Smoke test produced non-finite loss."
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            CONFIG.grad_clip_norm,
            error_if_nonfinite=True,
        )

        temp_optimizer.step()

        seen += 1

        print(
            f"  batch={seen}, loss={loss.item():.6f}"
        )

        if seen >= 2:
            break

    if seen < 2:
        raise RuntimeError(
            "Smoke test could not obtain two batches."
        )

    model.load_state_dict(backup)
    model.to(DEVICE)

    del backup, temp_optimizer

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Smoke test PASSED.")


smoke_test(model)


Running 2-batch smoke test...
  batch=1, loss=0.713601
  batch=2, loss=0.696132
Smoke test PASSED.


In [15]:

# ============================================================
# CELL 15 — TRAINING FUNCTION
# ============================================================

def train_stage(
    *,
    model: nn.Module,
    stage_name: str,
    epochs: int,
    lr: float,
    stage_dir: Path,
    resume: bool,
) -> dict:

    trainable = [
        p
        for p in model.parameters()
        if p.requires_grad
    ]

    if not trainable:
        raise RuntimeError(
            f"No trainable parameters for stage={stage_name}"
        )

    optimizer = torch.optim.AdamW(
        trainable,
        lr=lr,
        weight_decay=CONFIG.weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
    )

    last_ckpt = stage_dir / "last.ckpt"
    best_ckpt = stage_dir / "best.ckpt"

    history = []
    start_epoch = 1
    best_score = -float("inf")
    wait = 0

    if resume and CONFIG.resume and last_ckpt.exists():
        print(
            f"[{stage_name}] Resuming from {last_ckpt}"
        )

        ckpt = load_checkpoint(
            last_ckpt,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
        )

        history = list(
            ckpt.get("history", [])
        )

        start_epoch = (
            int(ckpt["epoch"]) + 1
        )

        best_score = float(
            ckpt.get(
                "best_metric_score",
                -float("inf"),
            )
        )

        if history:
            scores = [
                r.get(
                    "val_roc_auc",
                    float("nan"),
                )
                for r in history
            ]

            safe_scores = [
                s if np.isfinite(s)
                else -float("inf")
                for s in scores
            ]

            best_idx = int(
                np.argmax(safe_scores)
            )

            wait = max(
                0,
                len(history) - best_idx - 1,
            )

    if start_epoch > epochs:
        print(
            f"[{stage_name}] Stage already complete."
        )

        if not best_ckpt.exists():
            raise FileNotFoundError(
                f"Missing best checkpoint: {best_ckpt}"
            )

        return {
            "stage": stage_name,
            "history": history,
            "best_score": best_score,
            "best_checkpoint": str(best_ckpt),
            "last_checkpoint": str(last_ckpt),
        }

    for epoch in range(
        start_epoch,
        epochs + 1,
    ):

        start_time = time.time()

        train_out = run_epoch(
            model,
            loaders["train"],
            training=True,
            optimizer=optimizer,
        )

        val_out = run_epoch(
            model,
            loaders["val"],
            training=False,
        )

        train_metrics = binary_metrics(
            train_out["labels"],
            train_out["probabilities"],
            0.5,
        )

        val_metrics = binary_metrics(
            val_out["labels"],
            val_out["probabilities"],
            0.5,
        )

        monitor = val_metrics["roc_auc"]

        if not np.isfinite(monitor):
            monitor = val_metrics["f1"]

        scheduler.step(monitor)

        row = {
            "stage": stage_name,
            "epoch": int(epoch),
            "lr": float(
                optimizer.param_groups[0]["lr"]
            ),
            "train_loss": float(
                train_out["loss"]
            ),
            "val_loss": float(
                val_out["loss"]
            ),
            **{
                f"train_{k}": v
                for k, v in train_metrics.items()
            },
            **{
                f"val_{k}": v
                for k, v in val_metrics.items()
            },
            "epoch_seconds": float(
                time.time() - start_time
            ),
        }

        history.append(row)

        print(
            f"[{stage_name}] "
            f"epoch {epoch:02d}/{epochs} | "
            f"train_loss={row['train_loss']:.5f} | "
            f"val_loss={row['val_loss']:.5f} | "
            f"val_auc={row['val_roc_auc']:.5f} | "
            f"val_f1={row['val_f1']:.5f} | "
            f"lr={row['lr']:.2e}"
        )

        improved = (
            monitor > best_score + 1e-8
        )

        if improved:
            best_score = float(monitor)
            wait = 0
        else:
            wait += 1

        save_checkpoint(
            last_ckpt,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            best_score=best_score,
            history=history,
            stage_name=stage_name,
        )

        if improved:
            save_checkpoint(
                best_ckpt,
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                best_score=best_score,
                history=history,
                stage_name=stage_name,
            )

        atomic_csv_dump(
            pd.DataFrame(history),
            DIRS["metrics"]
            / f"{stage_name}_training_history.csv",
        )

        if wait >= CONFIG.patience:
            print(
                f"[{stage_name}] Early stopping."
            )
            break

    if not best_ckpt.exists():
        raise RuntimeError(
            f"No best checkpoint created for stage={stage_name}"
        )

    return {
        "stage": stage_name,
        "history": history,
        "best_score": best_score,
        "best_checkpoint": str(best_ckpt),
        "last_checkpoint": str(last_ckpt),
    }


In [ ]:

# ============================================================
# CELL 16 — STAGE 1: FROZEN BACKBONE
# ============================================================

for p in model.features.parameters():
    p.requires_grad = False

for p in model.classifier.parameters():
    p.requires_grad = True

print(
    "Trainable params (frozen):",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}",
)

frozen_summary = train_stage(
    model=model,
    stage_name="frozen",
    epochs=CONFIG.frozen_epochs,
    lr=CONFIG.frozen_lr,
    stage_dir=FROZEN_DIR,
    resume=True,
)

atomic_json_dump(
    frozen_summary,
    DIRS["metrics"]
    / "frozen_training_summary.json",
)

frozen_summary


Trainable params (frozen): 1,281


In [ ]:

# ============================================================
# CELL 17 — STAGE 2: FULL FINE-TUNING
# ============================================================

# Always initialize fine-tuning from frozen best checkpoint.
frozen_best = torch.load(
    FROZEN_DIR / "best.ckpt",
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(
    frozen_best["model_state_dict"]
)

model.to(DEVICE)

for p in model.parameters():
    p.requires_grad = True

print(
    "Trainable params (fine-tune):",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}",
)

# IMPORTANT:
# Do not resume a previously failed numerical fine-tuning attempt.
# We always start Stage 2 cleanly from the verified frozen best checkpoint.
for stale_name in ["last.ckpt", "best.ckpt"]:
    stale = FINETUNE_DIR / stale_name

    if stale.exists():
        print("Removing stale fine-tune checkpoint:", stale)
        stale.unlink()

history_file = (
    DIRS["metrics"]
    / "finetune_training_history.csv"
)

if history_file.exists():
    history_file.unlink()

summary_file = (
    DIRS["metrics"]
    / "finetune_training_summary.json"
)

if summary_file.exists():
    summary_file.unlink()

finetune_summary = train_stage(
    model=model,
    stage_name="finetune",
    epochs=CONFIG.finetune_epochs,
    lr=CONFIG.finetune_lr,
    stage_dir=FINETUNE_DIR,
    resume=False,
)

atomic_json_dump(
    finetune_summary,
    summary_file,
)

finetune_summary


In [ ]:

# ============================================================
# CELL 18 — VALIDATION THRESHOLD SELECTION
# ============================================================

FINAL_BEST_CKPT = (
    FINETUNE_DIR
    / "best.ckpt"
)

if not FINAL_BEST_CKPT.exists():
    raise FileNotFoundError(
        FINAL_BEST_CKPT
    )

best_final = torch.load(
    FINAL_BEST_CKPT,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(
    best_final["model_state_dict"]
)

model.to(DEVICE)
model.eval()

val_out = run_epoch(
    model,
    loaders["val"],
    training=False,
)

best_threshold, threshold_table = (
    select_threshold_on_validation(
        val_out["labels"],
        val_out["probabilities"],
    )
)

val_metrics_selected = binary_metrics(
    val_out["labels"],
    val_out["probabilities"],
    best_threshold,
)

atomic_csv_dump(
    threshold_table,
    DIRS["metrics"]
    / "validation_threshold_search.csv",
)

atomic_json_dump(
    val_metrics_selected,
    DIRS["metrics"]
    / "validation_selected_threshold_metrics.json",
)

print(
    "Validation-selected threshold:",
    best_threshold,
)

print(
    json.dumps(
        val_metrics_selected,
        indent=2,
    )
)


In [ ]:

# ============================================================
# CELL 19 — FINAL TEST
# ============================================================

test_out = run_epoch(
    model,
    loaders["test"],
    training=False,
)

test_metrics = binary_metrics(
    test_out["labels"],
    test_out["probabilities"],
    best_threshold,
)

test_predictions = pd.DataFrame({
    "sample_id": test_out["sample_ids"],
    "path": test_out["paths"],
    "label_int": test_out["labels"],
    "label": [
        INT_TO_LABEL[int(y)]
        for y in test_out["labels"]
    ],
    "prob_fake": test_out["probabilities"],
})

test_predictions["threshold"] = best_threshold

test_predictions["pred_int"] = (
    test_predictions["prob_fake"]
    .to_numpy()
    >= best_threshold
).astype(int)

test_predictions["prediction"] = (
    test_predictions["pred_int"]
    .map(INT_TO_LABEL)
)

test_predictions["correct"] = (
    test_predictions["label_int"]
    == test_predictions["pred_int"]
)

atomic_csv_dump(
    pd.DataFrame([
        {
            "model": "EfficientNet-B0",
            **test_metrics,
        }
    ]),
    DIRS["metrics"]
    / "final_test_metrics.csv",
)

atomic_csv_dump(
    test_predictions,
    DIRS["predictions"]
    / "test_predictions.csv",
)

print(
    json.dumps(
        test_metrics,
        indent=2,
    )
)


In [ ]:

# ============================================================
# CELL 20 — FIGURES
# ============================================================

def save_figure(
    fig,
    filename: str,
) -> Path:

    path = (
        DIRS["figures"]
        / filename
    )

    fig.savefig(
        path,
        dpi=150,
        bbox_inches="tight",
    )

    plt.close(fig)

    with Image.open(path) as img:
        if min(img.size) < 600:
            raise RuntimeError(
                f"Figure resolution too low: {img.size}"
            )

    return path


frozen_hist = pd.read_csv(
    DIRS["metrics"]
    / "frozen_training_history.csv"
)

finetune_hist = pd.read_csv(
    DIRS["metrics"]
    / "finetune_training_history.csv"
)

history_all = pd.concat(
    [
        frozen_hist,
        finetune_hist,
    ],
    ignore_index=True,
)

history_all["global_epoch"] = np.arange(
    1,
    len(history_all) + 1,
)

# Loss curve
fig, ax = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

ax.plot(
    history_all["global_epoch"],
    history_all["train_loss"],
    label="Training Loss",
)

ax.plot(
    history_all["global_epoch"],
    history_all["val_loss"],
    label="Validation Loss",
)

ax.set_title(
    "EfficientNet-B0 Training and Validation Loss"
)

ax.set_xlabel(
    "Global Epoch"
)

ax.set_ylabel(
    "Loss"
)

ax.legend()
ax.grid(
    True,
    alpha=0.25,
)

fig.tight_layout()

save_figure(
    fig,
    "training_validation_loss_curve.png",
)

# Validation metrics
fig, ax = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

ax.plot(
    history_all["global_epoch"],
    history_all["val_roc_auc"],
    label="Validation ROC-AUC",
)

ax.plot(
    history_all["global_epoch"],
    history_all["val_f1"],
    label="Validation F1",
)

ax.set_title(
    "EfficientNet-B0 Validation Metrics"
)

ax.set_xlabel(
    "Global Epoch"
)

ax.set_ylabel(
    "Score"
)

ax.set_ylim(
    0,
    1,
)

ax.legend()

ax.grid(
    True,
    alpha=0.25,
)

fig.tight_layout()

save_figure(
    fig,
    "validation_metrics_curve.png",
)

# ROC
y_test = test_out["labels"]
p_test = test_out["probabilities"]

fpr, tpr, _ = roc_curve(
    y_test,
    p_test,
)

roc_auc = roc_auc_score(
    y_test,
    p_test,
)

fig, ax = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

ax.plot(
    fpr,
    tpr,
    label=f"EfficientNet-B0 (AUC={roc_auc:.3f})",
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Chance",
)

ax.set_title(
    "Test ROC Curve"
)

ax.set_xlabel(
    "False Positive Rate"
)

ax.set_ylabel(
    "True Positive Rate"
)

ax.legend()

ax.grid(
    True,
    alpha=0.25,
)

fig.tight_layout()

save_figure(
    fig,
    "test_roc_curve.png",
)

# PR
precision_curve, recall_curve, _ = (
    precision_recall_curve(
        y_test,
        p_test,
    )
)

ap = average_precision_score(
    y_test,
    p_test,
)

fig, ax = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

ax.plot(
    recall_curve,
    precision_curve,
    label=f"EfficientNet-B0 (AP={ap:.3f})",
)

ax.set_title(
    "Test Precision-Recall Curve"
)

ax.set_xlabel(
    "Recall"
)

ax.set_ylabel(
    "Precision"
)

ax.legend()

ax.grid(
    True,
    alpha=0.25,
)

fig.tight_layout()

save_figure(
    fig,
    "test_precision_recall_curve.png",
)

# Confusion matrix
cm = confusion_matrix(
    y_test,
    (
        p_test >= best_threshold
    ).astype(int),
    labels=[0, 1],
)

fig, ax = plt.subplots(
    figsize=(8, 8),
    dpi=150,
)

im = ax.imshow(cm)
fig.colorbar(im, ax=ax)

ax.set_title(
    f"Test Confusion Matrix (Threshold={best_threshold:.3f})"
)

ax.set_xlabel(
    "Predicted Label"
)

ax.set_ylabel(
    "True Label"
)

ax.set_xticks(
    [0, 1],
    labels=["Real", "Fake"],
)

ax.set_yticks(
    [0, 1],
    labels=["Real", "Fake"],
)

for i in range(2):
    for j in range(2):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            fontsize=14,
        )

fig.tight_layout()

save_figure(
    fig,
    "test_confusion_matrix.png",
)

print("Figures saved.")


In [ ]:

# ============================================================
# CELL 21 — INFERENCE RELOAD TEST
# ============================================================

def inference_reload_test() -> dict:
    reloaded = build_model(
        pretrained=False
    ).to(DEVICE)

    ckpt = torch.load(
        FINAL_BEST_CKPT,
        map_location="cpu",
        weights_only=False,
    )

    reloaded.load_state_dict(
        ckpt["model_state_dict"]
    )

    reloaded.to(DEVICE)
    reloaded.eval()

    batch = next(
        iter(loaders["test"])
    )

    x = (
        batch["image"][
            : min(
                4,
                len(batch["image"]),
            )
        ]
        .to(DEVICE)
    )

    with torch.no_grad():
        probs = torch.sigmoid(
            reloaded(x)
            .squeeze(1)
            .float()
        )

    if probs.ndim != 1:
        raise RuntimeError(
            f"Unexpected inference shape: {tuple(probs.shape)}"
        )

    if not torch.isfinite(probs).all():
        raise FloatingPointError(
            "Reloaded model produced non-finite probabilities."
        )

    if ((probs < 0) | (probs > 1)).any():
        raise ValueError(
            "Probabilities outside [0,1]."
        )

    return {
        "status": "PASSED",
        "checkpoint": str(FINAL_BEST_CKPT),
        "n_samples": int(len(probs)),
        "probabilities": (
            probs
            .detach()
            .cpu()
            .numpy()
            .astype(float)
            .tolist()
        ),
    }


inference_test = inference_reload_test()

atomic_json_dump(
    inference_test,
    DIRS["metrics"]
    / "inference_reload_test.json",
)

print(
    json.dumps(
        inference_test,
        indent=2,
    )
)


In [ ]:

# ============================================================
# CELL 22 — FINAL MANIFEST + RUN SUMMARY
# ============================================================

def build_output_manifest(
    root: Path,
) -> pd.DataFrame:

    rows = []

    for p in sorted(
        root.rglob("*")
    ):
        if not p.is_file():
            continue

        size = p.stat().st_size

        digest = (
            sha256_file(p)
            if size <= 50 * 1024 * 1024
            else ""
        )

        rows.append({
            "relative_path": str(
                p.relative_to(root)
            ),
            "size_bytes": int(size),
            "sha256": digest,
        })

    return pd.DataFrame(rows)


manifest = build_output_manifest(
    RUN_DIR
)

atomic_csv_dump(
    manifest,
    RUN_DIR
    / "output_manifest.csv",
)

run_summary = {
    "run_id": RUN_ID,
    "experiment": "EfficientNet-B0 Eye ROI Transfer Learning Baseline",
    "model": "torchvision EfficientNet-B0",
    "pretrained_weights": "EfficientNet_B0_Weights.DEFAULT",
    "input": "combined eye ROI",
    "image_size": CONFIG.image_size,
    "seed": CONFIG.seed,
    "device": str(DEVICE),
    "metadata_path": str(METADATA_PATH),
    "roi_root": str(ROI_ROOT),
    "results_root": str(RESULTS_ROOT),
    "run_dir": str(RUN_DIR),
    "split_counts": data_accounting["split_class_counts"],
    "validation_selected_threshold": best_threshold,
    "validation_metrics": val_metrics_selected,
    "final_test_metrics": test_metrics,
    "inference_reload_test": inference_test["status"],
    "output_file_count": int(len(manifest)),
}

atomic_json_dump(
    run_summary,
    RUN_DIR
    / "run_summary.json",
)

print(
    json.dumps(
        run_summary,
        indent=2,
        ensure_ascii=False,
        default=str,
    )
)

print("\nFINAL OUTPUT DIRECTORY:")
print(RUN_DIR)
